# AQI Prediction Model
Predicts the air quality category (Good, Satisfactory, Moderate, Poor, Very Poor, Severe) from gas and weather readings.

## Step 1: Upload the Dataset
First upload `DATASET.csv` using the folder icon on the left, then run this cell.

## Step 2: Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, ConfusionMatrixDisplay)

## Step 3: Loading the Dataset

In [ ]:
df = pd.read_csv("DATASET.csv")
print("Rows, Columns:", df.shape)
df.head()

## Step 4: Dataset Overview (Summary and Missing Values)

In [ ]:
print("Missing values:", df.isnull().sum().sum())
df.describe().T

### Dropping rows with missing values
A few rows have missing values (including in the target `status`), which makes `train_test_split` fail. We remove them.

In [ ]:
before = len(df)
df = df.dropna().reset_index(drop=True)
print("Dropped rows:", before - len(df))
print("Missing values now:", df.isnull().sum().sum())
print("Rows, Columns:", df.shape)

## Step 5: AQI Category Distribution (Graph)

In [ ]:
order = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="status", order=order, hue="status",
              palette="viridis", legend=False)
plt.title("Distribution of AQI Categories")
plt.xlabel("AQI Category")
plt.ylabel("Number of Records")
plt.show()

## Step 6: Correlation Heatmap (Graph)

In [ ]:
cols = ["pm25", "pm10", "no2", "so2", "co", "o3", "nh3",
        "Temperature_C", "Humidity_%", "Wind_Speed_kmh"]
plt.figure(figsize=(10, 8))
sns.heatmap(df[cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Between Features")
plt.show()

## Step 7: Selecting Features (X) and Target (y)

In [ ]:
features = cols
X = df[features]
y = df["status"]

## Step 8: Train-Test Split (80% train, 20% test)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Training records:", len(X_train))
print("Testing records:", len(X_test))

## Step 9: Training the Model (Logistic Regression)
Logistic Regression works best when all features are on a similar scale, so we use a `Pipeline` that first scales the data (`StandardScaler`) and then trains the model.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

model = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000)),
])
model.fit(X_train, y_train)
print("Model trained!")

## Step 10: Evaluating the Model

In [ ]:
y_pred = model.predict(X_test)
print("Accuracy :", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred, average="macro", zero_division=0), 3))
print("Recall   :", round(recall_score(y_test, y_pred, average="macro", zero_division=0), 3))
print("F1 Score :", round(f1_score(y_test, y_pred, average="macro", zero_division=0), 3))
print()
print(classification_report(y_test, y_pred, zero_division=0))

## Step 11: Confusion Matrix (Graph)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, labels=order, cmap="Blues", ax=ax)
plt.title("Confusion Matrix")
plt.xticks(rotation=45)
plt.show()

## Step 12: Feature Importance (Graph)

In [ ]:
# Logistic Regression has no feature_importances_, so we use the average
# size of its coefficients (features are scaled, so they are comparable).
coef_size = np.abs(model.named_steps["clf"].coef_).mean(axis=0)
importance = pd.Series(coef_size / coef_size.sum(), index=features).sort_values()

plt.figure(figsize=(8, 6))
importance.plot(kind="barh", color="teal")
plt.title("Feature Importance (Logistic Regression coefficients)")
plt.show()

## Step 13: Creating the Prediction Function

In [ ]:
def predict_aqi(pm25, pm10, no2, so2, co, o3, nh3,
                temperature, humidity, wind_speed):
    reading = pd.DataFrame([[pm25, pm10, no2, so2, co, o3, nh3,
                             temperature, humidity, wind_speed]],
                           columns=features)
    status = model.predict(reading)[0]
    probs = dict(zip(model.classes_, model.predict_proba(reading)[0].round(3)))
    return status, probs

## Step 14: Predicting AQI for a New Place
Change the numbers below to any place's readings.

In [ ]:
status, probs = predict_aqi(pm25=65, pm10=120, no2=25, so2=10, co=1,
                            o3=30, nh3=76.8,
                            temperature=25, humidity=60, wind_speed=10)
print("Predicted AQI category:", status)
print(probs)

## Step 15: Saving the Model for Streamlit
This creates `model.joblib` and `model_meta.json`. Put both files in the same folder as `app.py`.

In [ ]:
import json
import joblib

joblib.dump(model, "model.joblib")

stats = {}
for f in features:
    s = df[f]
    stats[f] = {"min": float(s.min()), "max": float(s.max()),
                "median": float(s.median()),
                "p10": float(s.quantile(0.10)), "p90": float(s.quantile(0.90)),
                "p99": float(s.quantile(0.99))}

meta = {
    "features": features,
    "classes": list(model.classes_),
    "stats": stats,
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "macro_f1": float(f1_score(y_test, y_pred, average="macro", zero_division=0)),
    "report": classification_report(y_test, y_pred, zero_division=0, output_dict=True),
    "importance": {k: float(v) for k, v in importance.items()},
}
with open("model_meta.json", "w") as fh:
    json.dump(meta, fh, indent=2)

print("Saved model.joblib and model_meta.json")

# In Google Colab this downloads both files to your computer
try:
    from google.colab import files
    files.download("model.joblib")
    files.download("model_meta.json")
except ImportError:
    pass